# Demo 1: GoalScope: reading the next sub-task before it is written

**Insight.** Midway through a function with several sub-tasks, the LM already encodes which sub-task it will write
next, before it writes a single token of it. The goal heads carry that encoding, and GoalScope makes it readable: in
words, and as code in another language.

**Setup.** The prompt lists four sub-tasks. The code for the first three is prefilled, and the assistant turn stops at
`d =`, the start of the fourth. The last four prompt tokens (the newline, the indentation, `d` and `=`) are the
*pivots*. None of them says which sub-task comes next.

**Readout (GoalScope).** At a pivot, we sum what the 122 goal heads write into one vector. We add that vector to the
residual stream of a separate, neutral prompt, *"What are you doing at this line of the code?"*, and let the LM answer.
Whatever it describes must come from the vector. In a second test, the same vector is added to an empty C++ stub
`auto func (auto value){`. An LM judge (Claude) checks whether the answer describes (faithfulness) or the C++ body
implements (causality) the sub-task the LM actually writes next.

**Paper.** §3.2–3.3 (`fig:goalscope`), App. `app:goalscope`, App. `app:readout_sources` (`fig:heads-vs-residual`).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![GoalScope: sum the goal heads' outputs at one token and inject them into a neutral probe (§3.2, App. D).](figures/01_goalscope_method.png)

*GoalScope: sum the goal heads' outputs at one token and inject them into a neutral probe (§3.2, App. D).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# colors as in Demo 4: the LM's own code green, GoalScope-steered output purple, pivot token yellow
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    DST_C, PAT_C = 'sea_green3', 'plum2'
    DST_BG       = DST_C
    PIVOT_BG     = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    DST_C, PAT_C = '#38761d', '#5b277d'
    DST_BG       = '#d9ead3'
    PIVOT_BG     = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA         = '7'
MODEL        = 'Qwen/Qwen2.5-Coder-32B-Instruct'
GOAL_HEADS   = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'   # the 122 goal heads
SPEC_ID      = 'list[int]_python_016'   # a task of the evaluation corpus: 4 sub-tasks, the last one is next
ALPHA        = 0.1          # GoalScope strength: the probe's residual h gets + ALPHA * ||h|| along the vector
STEER_LAYERS = (35, 46)     # the probe is steered at layers 35..45
RESID_LAYERS = (40, 50)     # baseline: the residual stream, averaged over layers 40..49
SEED         = 123

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct and its 122 goal heads.

In [3]:
import torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape
from IPython.display import HTML, display
import html as _html

from src.models import ModelandTokenizer
from src.tokens import prepare_input
from src.utils import experiment_utils
from src.goals.head_sets import load_json_heads
from src.goals.goalscope import apply_goalscope, _build_causality_prompt, CAUSAL_PREFIX
from src.goals.goalscope_viz import goalscope_qualitative_html
from scripts.evaluation.eval_goalscope import (
    DEFAULT_DATASET, ALL_DATATYPES, MAX_NEW_TOKENS, GEN_MAX_NEW_TOKENS,
    load_locked_datapoints, build_prompt, generate_completion, frontier_impl_span,
    pivot_positions, hard_eval,
)
from scripts.evaluation.eval_readout_sources import residual_patchscope

console = Console(width=130, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
heads = load_json_heads(GOAL_HEADS, mt.n_layer, mt.n_heads)
gs_kwargs = dict(selected_heads=heads, strategy='additive_norm', alpha=ALPHA, steer_layer_range=STEER_LAYERS,
                 n_steer_tokens='all', max_new_tokens=MAX_NEW_TOKENS)
print(f'goal heads: {len(heads)} (layers {min(l for l, _ in heads)}-{max(l for l, _ in heads)})')

# the user turn's opening tag, to show the prompt from the instruction on
USER_TAG = '<|im_start|>user'
# GoalScope stops at the tokenizer's EOS only; cut a readout at the first chat-template turn tag
import re
trim = lambda s: re.split(r'<\|(?:user|observation|endoftext|im_end)\|>', s)[0]

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


goal heads: 122 (layers 28-55)


## 1. The task and the LM's own continuation
The prompt lists four sub-tasks, and the code for the first three is already written. The next sub-task, (4), is
highlighted, and the four pivots are in yellow. Then the LM writes the next line on its own. That line is what the
judge compares every readout against.

In [4]:
datapoints, _ = load_locked_datapoints(DEFAULT_DATASET, ALL_DATATYPES)
inst = dict(datapoints)[SPEC_ID]
prompt, var = build_prompt(mt, inst)
completion = generate_completion(mt, prompt, GEN_MAX_NEW_TOKENS)
full_text = prompt + completion
impl_span = frontier_impl_span(prompt, completion, var)       # the line the LM writes for sub-task (4)
pivs = pivot_positions(mt, prompt, full_text, ['prev_end', 'indent', 'var', 'eq'])
positions = sorted(pivs.values())
ids = prepare_input(prompts=full_text, tokenizer=mt)['input_ids'][0].tolist()
tok = lambda p: repr(mt.tokenizer.decode([ids[p]]))

t = Table(title='the four sub-tasks')
t.add_column('#'); t.add_column('sub-task'); t.add_column('status')
done = set(inst.src_progress)
for k, p in enumerate(inst.task.predicates, 1):
    nxt = p.id == inst.expected_next
    t.add_row(str(k), f'[bold {DST_C}]{escape(p.description)}[/]' if nxt else escape(p.description),
              'written' if p.id in done else f'[bold {DST_C}]next[/]' if nxt else 'later')
console.print(t)

def show_prompt(text, span, pivots, start_at):
    """the prompt from `start_at` on: `span` (chars) on DST_BG, pivot tokens on PIVOT_BG, the rest dim"""
    enc = prepare_input(prompts=text, tokenizer=mt, return_offsets_mapping=True)
    txt = Text()
    for i, (tid, (s, e)) in enumerate(zip(enc['input_ids'][0].tolist(), enc['offset_mapping'][0].tolist())):
        if e <= start_at: continue
        style = (f'bold black on {PIVOT_BG}' if i in pivots else
                 f'black on {DST_BG}' if span[0] <= s < span[1] else 'dim')
        txt.append(mt.tokenizer.decode([tid]), style=style)
    console.rule('the prompt'); console.print(txt)

i = prompt.index(inst.expected_next_desc)
next_line = (prompt.rfind('\n', 0, i) + 1, prompt.index('\n', i))
show_prompt(prompt, next_line, set(positions), start_at=prompt.index(USER_TAG))

stub = lambda p: p.rsplit('\n', 1)[-1]          # the prefilled `    d =`
console.rule(f'[bold {DST_C}]the LM writes the next line[/]')
console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{DST_C}]{escape(completion.rstrip())}[/]')

                          the four sub-tasks                           
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ # ┃ sub-task                                              ┃ status  ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ 1 │ the number of odd values                              │ written │
│ 2 │ the number of positive values                         │ written │
│ 3 │ whether the values are sorted in non-decreasing order │ written │
│ 4 │ whether any value appears more than once              │ next    │
└───┴───────────────────────────────────────────────────────┴─────────┘

─────────────────────────────────────────────────────────── the prompt ───────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given a list of integers `nums`, computes in sequential order:
(1) the number of odd values
(2) the number of positive values
(3) whether the values are sorted in non-decreasing order
(4) whether any value appears more than once
<|im_end|>
<|im_start|>assistant
<code>
def func(nums: list[int]):
    a = sum(1 for x in nums if x % 2 != 0)
    b = sum(1 for x in nums if x > 0)
    c = all(nums[i] <= nums[i+1] for i in range(len(nums)-1))
    d =

────────────────────────────────────────────────── the LM writes the next line ───────────────────────────────────────────────────

continuation:
    d = len(nums) != len(set(nums))
    return a, b, c, d
</code>

## 2. GoalScope at the pivots
At each pivot, the goal-head vector steers the neutral probe (faithfulness) and the empty C++ stub (causality). First,
the C++ stub with no vector, to see what it writes on its own. Then, per pivot, the description and the C++ body the
vector produces, each with the judge's verdict (YES = the same sub-task as the line the LM wrote above).

In [5]:
res = apply_goalscope(full_text, mt=mt, token_positions=positions, include_noncode=True, **gs_kwargs)
faith = {r['pos']: trim(r['patched_gen']) for r in res if r['scoped']}
causal_prompt = _build_causality_prompt(mt)
res_c = apply_goalscope(full_text, mt=mt, token_positions=positions, goalscope_prompt=causal_prompt, **gs_kwargs)
causal = {r['pos']: CAUSAL_PREFIX + trim(r['patched_gen']) for r in res_c}
default = trim(generate_completion(mt, causal_prompt, MAX_NEW_TOKENS))

V = lambda v: f'[bold green]{v}[/]' if v == 'YES' else f'[red]{v}[/]'
console.rule(f'[bold {DST_C}]C++ stub, no GoalScope[/]')
console.print(f'[grey50]{escape(CAUSAL_PREFIX)}[/][{DST_C}]{escape(default.rstrip())}[/]\n')

verdicts = {}
for a, p in pivs.items():
    vf = hard_eval(faith[p], impl_span, 'faithfulness')['verdict']
    vc = hard_eval(causal[p], impl_span, 'causality')['verdict']
    verdicts['goal heads', a] = (vf, vc)
    console.rule(f'[bold {PAT_C}]GoalScope at pivot {a} ({escape(tok(p))})[/]')
    console.print(f'[bold]GoalScope at the pivot:[/] [italic {PAT_C}]{escape(faith[p].strip())}[/]')
    console.print(f'[bold]judge:[/] {V(vf)}')
    console.print(f'[bold]C++ stub:[/]\n[grey50]{escape(CAUSAL_PREFIX)}[/][{PAT_C}]{escape(causal[p][len(CAUSAL_PREFIX):].rstrip())}[/]')
    console.print(f'[bold]judge:[/] {V(vc)}\n')

───────────────────────────────────────────────────── C++ stub, no GoalScope ─────────────────────────────────────────────────────

auto func (auto value){ return value * 2; }

────────────────────────────────────────────── GoalScope at pivot prev_end ('))\n') ──────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the current index is the last index or if there's a duplicate by comparing it to the set 
of the list.

judge: YES

C++ stub:
auto func (auto value){ return std::equal(value.begin(), value.end(), value.begin() + 1) && std::adjacent_find(value.begin(), 
value.end()) == value.end(); }

... (12 outputs left out)

GoalScope at the pivot: I'm checking if there are any duplicate elements in the list by comparing the length of the list with the 
length of the set created from the list. If they differ, duplicates exist.

judge: YES

C++ stub:
auto func (auto value){ 
    std::unordered_set<decltype(value)> seen;
    for(auto it = std::begin(value); it != std::end(value); ++it){
        if(!seen.insert(*it).second) return true;
    }
    return false;
}

judge: YES

## 3. GoalScope at every token the LM writes
The same readout at each token of the LM's continuation, while it writes the line and after it is done.

In [6]:
n_prompt = prepare_input(prompts=prompt, tokenizer=mt)['input_ids'].shape[1]     # the prompt ends at the `=` pivot
gen_positions = list(range(n_prompt, len(ids)))
res_gen = apply_goalscope(full_text, mt=mt, token_positions=gen_positions, include_noncode=True, **gs_kwargs)
gen_ro = {r['pos']: trim(r['patched_gen']) for r in res_gen if r['scoped']}

t = Table(title='GoalScope at every generated token')
t.add_column('token'); t.add_column('GoalScope readout (first 110 characters)', max_width=110)
for p in gen_positions:
    ro = ' '.join(gen_ro[p].split())
    t.add_row(escape(tok(p)), f'[{PAT_C}]' + escape(ro[:110] + ('…' if len(ro) > 110 else '')) + '[/]')
console.print(t)

                                              GoalScope at every generated token                                              
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ token     ┃ GoalScope readout (first 110 characters)                                                                       ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ' len'    │ I'm checking for duplicates by comparing the length of the list to the length of the set created from the      │
│           │ list…                                                                                                          │
│ '(nums'   │ I'm checking if there are any duplicate elements by comparing the length of the list to the length of the set  │
│           │ …                                                                                                              │
│ ')'       │ I'm checking if there are any duplicate elements by comparing the length of the list with the length of the    │
│           │ se…                                                                                                            │
│ ' !='     │ I'm checking for duplicates by comparing the length of the list with the length of the set of the list.        │
│ ' len'    │ I'm checking for duplicates by comparing the length of the list with the length of the set of the list.        │
│ '(set'    │ I'm checking for duplicates by comparing the length of the list to the length of the set of the list.          │
│ '(nums'   │ I'm checking for duplicates by comparing the length of the list to the length of the set of the list.          │
│ '))\n'    │ I'm checking if the length of the set of the list is equal to the length of the list itself. This determines   │
│           │ i…                                                                                                             │
│ '   '     │ I'm analyzing the specific line of code to determine its functionality and outputting the results in order:    │
│           │ is…                                                                                                            │
│ ' return' │ I'm analyzing the code to determine the value or state represented by each of the following four aspects: the  │
│           │ …                                                                                                              │
│ ' a'      │ I'm counting the number of elements in a list.                                                                 │
│ ','       │ I'm analyzing the code to determine the value of variables or conditions sequentially: first counting true     │
│           │ sta…                                                                                                           │
│ ' b'      │ I'm checking the current line of code for specific conditions: whether it's executable, contains a condition,  │
│           │ …                                                                                                              │
│ ','       │ I'm checking if the current line of code is true or false.                                                     │
│ ' c'      │ I'm checking if the current line of code is executing without errors and if it's reaching the end of the line  │
│           │ …                                                                                                              │
│ ','       │ I'm checking the current line's action in the code.                                                            │
│ ' d'      │ Sure, please provide the line of code.                                                                         │
│ '\n'      │ Sure, here's a concise function that does what you're asking for: ```python def describe_line_of_code(code):   │
│           │ r…                                               

### Paper-style view
The viewer of the paper's qualitative figures: hover or click a token to see its readout. The four pivots are pinned.
It is also saved as a standalone page in `notebooks/demos/outputs/` (it needs JavaScript; on GitHub, use the table
above).

In [7]:
import warnings; warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame')
# merge the two runs (pivots + generated tokens are scoped, the rest is display-only context),
# show the instruction + code (drop the system-prompt tokens), pin the four pivots
scoped = {r['pos']: r for r in res + res_gen if r['scoped']}
merged = [scoped.get(r['pos'], r) for r in res]
start_char = full_text.index(USER_TAG)
offs = prepare_input(prompts=full_text, tokenizer=mt, return_offsets_mapping=True)['offset_mapping'][0].tolist()
start_tok = next(i for i, (s, e) in enumerate(offs) if e > start_char)
rows = [r for r in merged if r['pos'] >= start_tok]
os.makedirs('notebooks/demos/outputs', exist_ok=True)
doc = goalscope_qualitative_html(rows, title=f'GoalScope · {SPEC_ID} · {MODEL.split("/")[-1]} · {len(heads)} goal heads',
                                 save_path='notebooks/demos/outputs/01_goalscope_viewer'
                                 + ('' if MODEL.startswith('Qwen/Qwen2.5-Coder') else '_' + MODEL.split('/')[-1]) + '.html',
                                 initial_pins=positions)
display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" style="width:100%;height:560px;'
             f'border:1px solid #e5e7eb;border-radius:10px;" sandbox="allow-scripts"></iframe>'))

## 4. Baseline: the residual stream at the same pivots
Is any hidden state at the pivot enough? The same two tests, but steering with the residual stream (averaged over
layers 40–49) instead of the goal heads' write. The residual stream also carries the current token, its position and
the syntax around it, and that drowns out the sub-task.

In [8]:
src_layers = list(range(*RESID_LAYERS))
r_f = residual_patchscope(mt, full_text, positions, src_layers, alpha=ALPHA, source='resid')
r_c = residual_patchscope(mt, full_text, positions, src_layers, alpha=ALPHA, source='resid', probe_prompt=causal_prompt)
resid_faith = {r['pos']: trim(r['patched_gen']) for r in r_f}
resid_causal = {r['pos']: CAUSAL_PREFIX + trim(r['patched_gen']) for r in r_c}

one = lambda s, n=60: escape(' '.join(s.split())[:n] + ('…' if len(' '.join(s.split())) > n else ''))
t = Table(title=f'residual stream (layers {RESID_LAYERS[0]}-{RESID_LAYERS[1] - 1}) at the pivots', show_lines=True)
t.add_column('pivot'); t.add_column('description'); t.add_column('judge'); t.add_column('C++ stub'); t.add_column('judge')
for a, p in pivs.items():
    vf = hard_eval(resid_faith[p], impl_span, 'faithfulness')['verdict']
    vc = hard_eval(resid_causal[p], impl_span, 'causality')['verdict']
    verdicts['residual', a] = (vf, vc)
    t.add_row(a, one(resid_faith[p]), V(vf), one(resid_causal[p]), V(vc))
console.print(t)

t = Table(title='judge verdicts per pivot (faithfulness / causality)')
t.add_column('readout source')
for a in pivs: t.add_column(a)
for src, name in [('goal heads', f'{len(heads)} goal heads'), ('residual', f'residual, layers {RESID_LAYERS[0]}-{RESID_LAYERS[1] - 1}')]:
    t.add_row(name, *[f'{V(verdicts[src, a][0])} / {V(verdicts[src, a][1])}' for a in pivs])
console.print(t)

                                           residual stream (layers 40-49) at the pivots                                           
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ pivot    ┃ description                                 ┃ judge      ┃ C++ stub                                    ┃ judge      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ prev_end │ I'm checking the                            │ NO_INFO    │ auto func (auto value){ return value;}      │ NO         │
├──────────┼─────────────────────────────────────────────┼────────────┼─────────────────────────────────────────────┼────────────┤
│ indent   │ I return unique elements return set return  │ YES        │ auto func (auto value){ return std:: return │ HEAVY_LOOP │
│          │ len unique_elemen…                          │            │ std:: return std…                           │            │
├──────────┼─────────────────────────────────────────────┼────────────┼─────────────────────────────────────────────┼────────────┤
│ var      │ Executing = = ( len > e = = ( len > = = = = │ HEAVY_LOOP │ auto func (auto value){ return value > = (  │ HEAVY_LOOP │
│          │ = = = = = = d = …                           │            │ value > = ( value…                          │            │
├──────────┼─────────────────────────────────────────────┼────────────┼─────────────────────────────────────────────┼────────────┤
│ eq       │ I len len len len len len len len len len   │ HEAVY_LOOP │ auto func (auto value){ std std any len len │ HEAVY_LOOP │
│          │ len len len len le…                         │            │ len len len len …                           │            │
└──────────┴─────────────────────────────────────────────┴────────────┴─────────────────────────────────────────────┴────────────┘

                              judge verdicts per pivot (faithfulness / causality)                               
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ readout source         ┃ prev_end     ┃ indent           ┃ var                     ┃ eq                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 122 goal heads         │ YES / YES    │ YES / YES        │ YES / YES               │ YES / YES               │
│ residual, layers 40-49 │ NO_INFO / NO │ YES / HEAVY_LOOP │ HEAVY_LOOP / HEAVY_LOOP │ HEAVY_LOOP / HEAVY_LOOP │
└────────────────────────┴──────────────┴──────────────────┴─────────────────────────┴─────────────────────────┘

## What to expect
<img src="figures/01_goalscope_results.png" width="400" alt="GoalScope readout scores per pivot and baselines (Figure 2, right, in the paper)">

At every pivot, the goal-head readout should describe the next sub-task (here, checking for duplicate values), and the
steered C++ stub should implement it. The residual-stream readouts mostly loop or say nothing. Across 253 tasks, the
judge accepts the goal-head readout 85% of the time for faithfulness and 75% for causality (mean over the four pivots;
97% / 96% at `=`, 59% / 52% at the newline that ends the previous line). Steering with the whole attention output of
layers 40–50 gets 33% / 29%, and the residual stream about 1%. See §3.2–3.3 and Figure 2 in the paper; App. D
describes GoalScope, and App. E (Figure 12) compares readout sources.